# vizcraft — Skyline & Dumbbell

My two best charts from the **tallest-buildings** dataset, rendered with the
zero-dependency [`vizcraft`](https://github.com/simran587/visualization) library.

**Run order:** Runtime → **Restart session**, then Runtime → **Run all**.

## 1. Install vizcraft (pip)

`--force-reinstall --no-cache-dir` pulls the **latest** code from the branch
(plain `pip install` skips it because the package looks 'already installed').

> **Seeing old charts / a `highlight_color` error?** You have a stale install.
> Do **Runtime → Restart session**, then **Run all**. If you opened a *saved
> copy* in Drive, reopen the notebook from the GitHub link instead.

If the repo is **private**, this install fails silently \u2014 make it public or
append a token: `git+https://<TOKEN>@github.com/...`.

In [ ]:
!pip install --force-reinstall --no-deps --no-cache-dir \
    "git+https://github.com/simran587/visualization.git@claude/visualization-library-dataset-ei5m7t"

## 2. Verify the version, then load the data

This should print **0.1.1** and confirm the new options exist. If it raises,
restart the runtime (see the note above).

In [ ]:
import inspect, vizcraft as vc
from IPython.display import SVG, display

print('vizcraft version:', vc.__version__)
assert 'highlight_color' in inspect.signature(vc.dumbbell_chart).parameters, (
    'Old vizcraft is still cached \u2192 Runtime > Restart session, then Run all.')
print('OK \u2014 latest code loaded.')

data = vc.load_tallest_buildings()
print(len(data), 'buildings ·', data.columns)
data.head()      # first 5 rows, rendered as a table

## 3. Skyline — the 15 tallest, to scale

Each building is drawn as its own recognizable silhouette, scaled to its true
height, with Burj Khalifa highlighted in **violet**.

In [ ]:
top = data.top('height_m', 15)

skyline = vc.skyline_chart(
    top.column('building'), top.column('height_m'),
    title='Burj Khalifa still towers far above the next tallest',
    subtitle='The 15 tallest buildings, each drawn to scale as its own silhouette \u00b7 metres',
    highlight='Burj Khalifa', unit=' m',
)
skyline.save('skyline.svg')
display(SVG(skyline.to_string()))

## 4. Dumbbell — each country's shortest-to-tallest range

A horizontal dumbbell (open \u25cb = shortest, filled \u25cf = tallest) with the UAE
\u2014 the widest range \u2014 highlighted in **maroon** (a different accent from the skyline).

In [ ]:
cats, lo, hi = [], [], []
for country, ds in data.groups('country').items():
    heights = ds.column('height_m')
    if len(heights) >= 2:
        cats.append(country); lo.append(min(heights)); hi.append(max(heights))
order = sorted(range(len(cats)), key=lambda i: hi[i], reverse=True)

dumbbell = vc.dumbbell_chart(
    [cats[i] for i in order], [lo[i] for i in order], [hi[i] for i in order],
    highlight='United Arab Emirates', highlight_color='#8c1c2e',   # maroon
    title='The UAE spans the widest range of any country here',
    subtitle='Shortest (\u25cb) to tallest (\u25cf) building per country, in metres',
    unit=' m',
)
dumbbell.save('dumbbell.svg')
display(SVG(dumbbell.to_string()))

## 5. (Optional) dark theme

Every chart takes `theme="dark"`.

In [ ]:
display(SVG(vc.skyline_chart(
    top.column('building'), top.column('height_m'),
    title='Burj Khalifa still towers far above the next tallest',
    subtitle='At night \u00b7 metres', highlight='Burj Khalifa', unit=' m',
    theme='dark',
).to_string()))